In [1]:
import requests
import json

# Specify the search term
query = '"generative ai"'

# Define the API endpoint URL
url = "http://api.semanticscholar.org/graph/v1/paper/search/bulk"

# Define the query parameters
query_params = {
    "query": '"carnivores pleistocene"',
    "fields": "title,url,publicationTypes,publicationDate,openAccessPdf",
    "year": "2020-"
}


# Send the API request
response = requests.get(url, params=query_params).json()

In [6]:
import requests
import json

query = "pleistocene + (carnivores | carnivore)"
fields = "title,year"

url = f"http://api.semanticscholar.org/graph/v1/paper/search/bulk?query={query}&fields={fields}&year=2023-"
r = requests.get(url).json()

print(f"Will retrieve an estimated {r['total']} documents")
retrieved = 0

with open(f"papers.jsonl", "a") as file:
    while True:
        if "data" in r:
            retrieved += len(r["data"])
            print(f"Retrieved {retrieved} papers...")
            for paper in r["data"]:
                print(json.dumps(paper), file=file)
        if "token" not in r:
            break
        r = requests.get(f"{url}&token={r['token']}").json()

print(f"Done! Retrieved {retrieved} papers total")

Will retrieve an estimated 37 documents
Retrieved 37 papers...
Done! Retrieved 37 papers total


In [19]:
import requests
import json

query = "pleistocene carnivores"
fields = "title,year,abstract,externalIds,url,isOpenAccess,openAccessPdf,fieldsOfStudy"

url = f"http://api.semanticscholar.org/graph/v1/paper/search/bulk?query={query}&fields={fields}&year=-2025"
r = requests.get(url).json()

print(f"Will retrieve an estimated {r['total']} documents")
retrieved = 0

with open(f"papers_5.jsonl", "a") as file:
    while True:
        if "data" in r:
            retrieved += len(r["data"])
            print(f"Retrieved {retrieved} papers...")
            for paper in r["data"]:
                print(json.dumps(paper), file=file)
        if "token" not in r:
            break
        r = requests.get(f"{url}&token={r['token']}").json()

print(f"Done! Retrieved {retrieved} papers total")

Will retrieve an estimated 854 documents
Retrieved 854 papers...
Done! Retrieved 854 papers total


In [18]:
with open(f"papers_4.jsonl", "a") as file:
    while True:
        if "data" in r:
            retrieved += len(r["data"])
            print(f"Retrieved {retrieved} papers...")
            for paper in r["data"]:
                print(json.dumps(paper), file=file)
        if "token" not in r:
            break
        r = requests.get(f"{url}&token={r['token']}").json()

print(f"Done! Retrieved {retrieved} papers total")

Done! Retrieved 854 papers total


In [14]:
import requests
import json

query = "pleistocene carnivores"
fields = "title,year,abstract"

url = f"http://api.semanticscholar.org/graph/v1/paper/search?query={query}&year=-2025"
r = requests.get(url).json()

print(f"Will retrieve an estimated {r['total']} documents")
retrieved = 0

with open(f"papers_3.jsonl", "a") as file:
    while True:
        if "data" in r:
            retrieved += len(r["data"])
            print(f"Retrieved {retrieved} papers...")
            for paper in r["data"]:
                print(json.dumps(paper), file=file)
        if "token" not in r:
            break
        r = requests.get(f"{url}&token={r['token']}").json()

print(f"Done! Retrieved {retrieved} papers total")

Will retrieve an estimated 103761 documents
Retrieved 10 papers...
Done! Retrieved 10 papers total


In [1]:
import pandas as pd

# Load JSONL file into a DataFrame
df = pd.read_json('jsons/papers_5.jsonl', lines=True)


In [14]:
df.columns

Index(['paperId', 'externalIds', 'url', 'title', 'abstract', 'year',
       'isOpenAccess', 'openAccessPdf', 'fieldsOfStudy'],
      dtype='object')

In [2]:
df["isOpenAccess"].value_counts()

isOpenAccess
False    593
True     261
Name: count, dtype: int64

In [12]:
n = 0
for index,row in df.iterrows():
    if "CorpusId" not in row["externalIds"].keys():
        print(row["externalIds"].keys())

In [13]:
df.iloc[100]["externalIds"]

{'MAG': '1589137133', 'CorpusId': 82219643}

In [28]:
import os
import requests

# Ensure the directory for storing PDFs exists
output_dir = "pdf"
os.makedirs(output_dir, exist_ok=True)

# Iterate over the DataFrame and download each PDF
for i, r in df[df["isOpenAccess"] == True].iterrows():
    try:
        url = r["openAccessPdf"]["url"]
        if url:
            print(f"Downloading: {url}")
            # Fetch the PDF
            response = requests.get(url, stream=True)
            response.raise_for_status()  # Ensure request was successful
            
            # Save the file
            pdf_name = os.path.join(output_dir, f"document_{i}.pdf")
            with open(pdf_name, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    pdf_file.write(chunk)
            print(f"Saved to: {pdf_name}")
        else:
            print(f"No URL for index {i}")
    except Exception as e:
        print(f"Error processing index {i}: {e}")


Downloading: https://royalsocietypublishing.org/doi/pdf/10.1098/rsbl.2016.0062
Error processing index 1: 403 Client Error: Forbidden for url: https://royalsocietypublishing.org/doi/pdf/10.1098/rsbl.2016.0062
Downloading: https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0092144&type=printable
Saved to: pdf/document_4.pdf
Downloading: https://www.tandfonline.com/doi/pdf/10.1080/08912963.2018.1429856?needAccess=true
Error processing index 13: 403 Client Error: Forbidden for url: https://www.tandfonline.com/doi/pdf/10.1080/08912963.2018.1429856?needAccess=true
Downloading: http://palaeo-electronica.org/content/pdfs/464.pdf
Saved to: pdf/document_19.pdf
Downloading: https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0163591&type=printable
Saved to: pdf/document_25.pdf
Downloading: https://ddd.uab.cat/pub/artpub/2022/250226/jouhumevo_a2022v162a103108.pdf
Saved to: pdf/document_26.pdf
Downloading: http://journals.openedition.org/quaternaire/pdf/969
Er

In [ ]:
import os
import requests

# Directories
pdf_dir = "pdf/txt"

# Check which files are already downloaded
downloaded_files = {f for f in os.listdir(pdf_dir) if f.endswith(".txt")}

# Iterate through the DataFrame to reconcile and rename
for i, r in df[df["isOpenAccess"] == True].iterrows():
    try:
        url = r["openAccessPdf"]["url"]
        name = r["title"]  # Assuming the DataFrame has a column 'name' for filenames
        
        # Construct meaningful file name
        clean_name = "".join(c if c.isalnum() or c in (' ', '_', '-') else '_' for c in name)
        file_name = f"{clean_name}_{i}.txt"
        file_path = os.path.join(pdf_dir, file_name)
        
        # Check if this PDF is already downloaded
        if any(f.startswith(f"document_{i}") for f in downloaded_files):
            # Rename the generic file to the meaningful name
            generic_file = [f for f in downloaded_files if f.startswith(f"document_{i}")][0]
            os.rename(os.path.join(pdf_dir, generic_file), file_path)
            print(f"Renamed {generic_file} to {file_name}")
        elif not os.path.exists(file_path):
            # If not downloaded, retry download
            print(f"Retrying download for: {url}")
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            # Save the file with the meaningful name
            with open(file_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    pdf_file.write(chunk)
            print(f"Downloaded and saved: {file_name}")
    except Exception as e:
        print(f"Error processing index {i} (name: {name}): {e}")


Renamed document_165.txt to Ancient mitochondrial DNA reveals convergent evolution of giant short-faced bears _Tremarctinae_ in North and South America_1.txt
Renamed document_462.txt to Bone Accumulation by Leopards in the Late Pleistocene in the Moncayo Massif _Zaragoza_ NE Spain__4.txt
Renamed document_132.txt to General to specific Quaternary taphonomy_13.txt
Renamed document_192.txt to A revised listing of fossil mammals from the Haasgat cave system ex situ deposits _HGD__ South Africa_19.txt
Renamed document_252.txt to Under the Skin of a Lion_ Unique Evidence of Upper Paleolithic Exploitation and Use of Cave Lion _Panthera spelaea_ from the Lower Gallery of La Garma _Spain__25.txt
Renamed document_268.txt to A comparative study of the Early Pleistocene carnivore guild from Dmanisi _Georgia___26.txt
Renamed document_278.txt to THE PLIOCENE-PLEISTOCENE BOUNDARY_ WHICH SIGNIFICANCE FOR THE SO CALLED _WOLF EVENT__ EVIDENCES FROM WESTERN EUROPE_27.txt
Renamed document_28.txt to Late P

: 

In [30]:
import os
from docling.document_converter import DocumentConverter

pdf_dir = "pdf"
txt_dir = os.path.join(pdf_dir, "txt")
os.makedirs(txt_dir, exist_ok=True)
converter = DocumentConverter()

for pdf_file in os.listdir(pdf_dir):
    pdf_path = os.path.join(pdf_dir, pdf_file)
    
    if pdf_file.endswith(".pdf") and os.path.isfile(pdf_path):
        try:
            result = converter.convert(pdf_path)
            markdown_text = result.document.export_to_markdown()
            txt_file_path = os.path.join(txt_dir, f"{os.path.splitext(pdf_file)[0]}.txt")
            with open(txt_file_path, "w", encoding="utf-8") as txt_file:
                txt_file.write(markdown_text)
        except Exception as e:
            print(f"Error processing {pdf_path}: {e}")


/Users/asdls/miniconda3/envs/scholarly/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 107240.73it/s]


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
Error processing pdf/document_74.pdf: 


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
ERR#: COULD NOT CONVERT TO RS THIS TABLE TO COMPUTE SPANS


2024-11-15 21:07:21.899 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-selected
2024-11-15 21:07:21.899 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2024-11-15 21:07:21.900 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2024-11-15 21:07:21.900 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2024-11-15 21:07:21.900 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2024-11-15 21:07:21.900 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2024-11-15 21:07:21.900 (6105.784s) [         15EDD05]    doc_normalisation.h:448   WARN| found new `other` type: checkbox-unselected
2024-11-15 21:07:21.900 (6105.784s) [         15EDD05]    doc_no

Error processing pdf/document_182.pdf: 


*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
*ERR* --- *ERR*
*ERR* Table is not square! *ERR*
*Padding to square...*
